In [0]:
# Bronze to Silver Transformation - Energy Portfolio Data
# Author: Genie Code Assistant
# Date: 2026-08-29

import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BRONZE_CATALOG = "portfolio_energia.bronze"
SILVER_CATALOG = "portfolio_energia.silver"

In [0]:
%sql
-- Create the silver schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS portfolio_energia.silver
COMMENT 'Silver layer for curated energy portfolio data';

SHOW SCHEMAS IN portfolio_energia;

databaseName
bronze
default
information_schema
silver


In [0]:
# ===================================================================
# Transformation 1: ccee_pld_historico_semanal -> pld_historico_semanal
# ===================================================================

# Read bronze table
df_bronze = spark.table(f"{BRONZE_CATALOG}.ccee_pld_historico_semanal")

# Transform to pandas for easier manipulation
df_pd = df_bronze.toPandas()

# Data transformations
df_silver = df_pd.copy()

# 1. Parse MES_REFERENCIA (YYYYMM) to date (first day of month)
df_silver['mes_referencia'] = pd.to_datetime(df_silver['MES_REFERENCIA'], format='%Y%m', errors='coerce')

# 2. Create full date by combining mes_referencia and DIA
# Extract year and month from mes_referencia, then add the day
df_silver['dia_inteiro'] = pd.to_numeric(df_silver['DIA'], errors='coerce')
df_silver['data'] = df_silver.apply(
    lambda row: pd.Timestamp(year=row['mes_referencia'].year, 
                             month=row['mes_referencia'].month, 
                             day=int(row['dia_inteiro'])) 
    if pd.notna(row['mes_referencia']) and pd.notna(row['dia_inteiro']) else pd.NaT, 
    axis=1
)

# 3. Convert HORA to integer
df_silver['hora'] = pd.to_numeric(df_silver['HORA'], errors='coerce').astype('Int64')

# 4. Convert PLD_HORA to float
df_silver['pld_hora'] = pd.to_numeric(df_silver['PLD_HORA'], errors='coerce')

# 5. Standardize SUBMERCADO
df_silver['submercado'] = df_silver['SUBMERCADO'].str.strip().str.upper()

# 6. Parse PERIODO_COMERCIALIZACAO
df_silver['periodo_comercializacao'] = pd.to_numeric(df_silver['PERIODO_COMERCIALIZACAO'], errors='coerce').astype('Int64')

# Data Quality: Remove invalid records
initial_count = len(df_silver)

# Remove rows with null essential fields
df_silver = df_silver[
    df_silver['data'].notna() &
    df_silver['hora'].notna() &
    df_silver['pld_hora'].notna() &
    df_silver['submercado'].notna()
]

# Validate HORA range (0-23)
df_silver = df_silver[df_silver['hora'].between(0, 23)]

# Validate positive PLD values
df_silver = df_silver[df_silver['pld_hora'] >= 0]

# Remove duplicates based on primary key
df_silver = df_silver.drop_duplicates(subset=['submercado', 'data', 'hora'], keep='first')

# Select and rename columns for silver layer
df_silver_final = df_silver[[
    'submercado',
    'mes_referencia',
    'periodo_comercializacao',
    'data',
    'hora',
    'pld_hora'
]].copy()

# Add audit columns
df_silver_final['processed_timestamp'] = datetime.now()
df_silver_final['data_quality_flag'] = 'VALID'

# Convert back to Spark DataFrame and write to silver
df_spark_silver = spark.createDataFrame(df_silver_final)

# Write to silver layer
df_spark_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_CATALOG}.pld_historico_semanal")

In [0]:
# ===================================================================
# Transformation 2: ccee_pld_horario -> pld_horario
# ===================================================================

# Read bronze table
df_bronze = spark.table(f"{BRONZE_CATALOG}.ccee_pld_horario")

# Transform to pandas
df_pd = df_bronze.toPandas()

# Data transformations
df_silver = df_pd.copy()

# 1. Parse MES_REFERENCIA (YYYYMM) to date
df_silver['mes_referencia'] = pd.to_datetime(df_silver['MES_REFERENCIA'], format='%Y%m', errors='coerce')

# 2. Create full date
df_silver['dia_inteiro'] = pd.to_numeric(df_silver['DIA'], errors='coerce')
df_silver['data'] = df_silver.apply(
    lambda row: pd.Timestamp(year=row['mes_referencia'].year, 
                             month=row['mes_referencia'].month, 
                             day=int(row['dia_inteiro'])) 
    if pd.notna(row['mes_referencia']) and pd.notna(row['dia_inteiro']) else pd.NaT, 
    axis=1
)

# 3. Convert HORA to integer
df_silver['hora'] = pd.to_numeric(df_silver['HORA'], errors='coerce').astype('Int64')

# 4. Convert PLD_HORA to float
df_silver['pld_hora'] = pd.to_numeric(df_silver['PLD_HORA'], errors='coerce')

# 5. Standardize SUBMERCADO
df_silver['submercado'] = df_silver['SUBMERCADO'].str.strip().str.upper()

# 6. Parse PERIODO_COMERCIALIZACAO
df_silver['periodo_comercializacao'] = pd.to_numeric(df_silver['PERIODO_COMERCIALIZACAO'], errors='coerce').astype('Int64')

# 7. Keep ano_arquivo for reference
df_silver['ano_arquivo'] = df_silver['_ano_arquivo']

# Data Quality
initial_count = len(df_silver)

df_silver = df_silver[
    df_silver['data'].notna() &
    df_silver['hora'].notna() &
    df_silver['pld_hora'].notna() &
    df_silver['submercado'].notna()
]

df_silver = df_silver[df_silver['hora'].between(0, 23)]
df_silver = df_silver[df_silver['pld_hora'] >= 0]
df_silver = df_silver.drop_duplicates(subset=['submercado', 'data', 'hora'], keep='first')

# Select final columns
df_silver_final = df_silver[[
    'submercado',
    'mes_referencia',
    'periodo_comercializacao',
    'data',
    'hora',
    'pld_hora',
    'ano_arquivo'
]].copy()

df_silver_final['processed_timestamp'] = datetime.now()
df_silver_final['data_quality_flag'] = 'VALID'

# Write to silver
df_spark_silver = spark.createDataFrame(df_silver_final)

df_spark_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_CATALOG}.pld_horario")

In [0]:
# ===================================================================
# Transformation 3: ons_balanco_energia_subsistema -> balanco_energia_subsistema
# ===================================================================

# Read bronze table
df_bronze = spark.table(f"{BRONZE_CATALOG}.ons_balanco_energia_subsistema")

# Transform to pandas
df_pd = df_bronze.toPandas()

# Data transformations
df_silver = df_pd.copy()

# 1. Parse din_instante to timestamp
df_silver['timestamp'] = pd.to_datetime(df_silver['din_instante'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# 2. Convert all generation/load values to float
value_columns = ['val_gerhidraulica', 'val_gertermica', 'val_gereolica', 'val_gersolar', 'val_carga', 'val_intercambio']

for col in value_columns:
    df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

# 3. Rename columns for clarity
df_silver = df_silver.rename(columns={
    'id_subsistema': 'subsistema_id',
    'nom_subsistema': 'subsistema_nome',
    'val_gerhidraulica': 'geracao_hidraulica_mw',
    'val_gertermica': 'geracao_termica_mw',
    'val_gereolica': 'geracao_eolica_mw',
    'val_gersolar': 'geracao_solar_mw',
    'val_carga': 'carga_mw',
    'val_intercambio': 'intercambio_mw'
})

# 4. Standardize subsistema_id
df_silver['subsistema_id'] = df_silver['subsistema_id'].str.strip().str.upper()

# Data Quality
initial_count = len(df_silver)

# Remove rows with null timestamp or subsistema_id
df_silver = df_silver[
    df_silver['timestamp'].notna() &
    df_silver['subsistema_id'].notna()
]

# Fill NaN values in generation columns with 0 (represents no generation)
generation_cols = ['geracao_hidraulica_mw', 'geracao_termica_mw', 'geracao_eolica_mw', 'geracao_solar_mw']
for col in generation_cols:
    df_silver[col] = df_silver[col].fillna(0)

# Validate non-negative values for generation (can't be negative)
for col in generation_cols:
    df_silver = df_silver[df_silver[col] >= 0]

# Remove duplicates
df_silver = df_silver.drop_duplicates(subset=['subsistema_id', 'timestamp'], keep='first')

# 5. Add calculated fields
df_silver['geracao_total_mw'] = (
    df_silver['geracao_hidraulica_mw'] + 
    df_silver['geracao_termica_mw'] + 
    df_silver['geracao_eolica_mw'] + 
    df_silver['geracao_solar_mw']
)

# Calculate generation mix percentages (avoid division by zero)
df_silver['pct_hidraulica'] = df_silver.apply(
    lambda row: (row['geracao_hidraulica_mw'] / row['geracao_total_mw'] * 100) if row['geracao_total_mw'] > 0 else 0,
    axis=1
)
df_silver['pct_termica'] = df_silver.apply(
    lambda row: (row['geracao_termica_mw'] / row['geracao_total_mw'] * 100) if row['geracao_total_mw'] > 0 else 0,
    axis=1
)
df_silver['pct_eolica'] = df_silver.apply(
    lambda row: (row['geracao_eolica_mw'] / row['geracao_total_mw'] * 100) if row['geracao_total_mw'] > 0 else 0,
    axis=1
)
df_silver['pct_solar'] = df_silver.apply(
    lambda row: (row['geracao_solar_mw'] / row['geracao_total_mw'] * 100) if row['geracao_total_mw'] > 0 else 0,
    axis=1
)

# 6. Keep ano_arquivo for reference
df_silver['ano_arquivo'] = df_silver['_ano_arquivo']

# Select final columns
df_silver_final = df_silver[[
    'subsistema_id',
    'subsistema_nome',
    'timestamp',
    'geracao_hidraulica_mw',
    'geracao_termica_mw',
    'geracao_eolica_mw',
    'geracao_solar_mw',
    'geracao_total_mw',
    'pct_hidraulica',
    'pct_termica',
    'pct_eolica',
    'pct_solar',
    'carga_mw',
    'intercambio_mw',
    'ano_arquivo'
]].copy()

df_silver_final['processed_timestamp'] = datetime.now()
df_silver_final['data_quality_flag'] = 'VALID'

# Write to silver
df_spark_silver = spark.createDataFrame(df_silver_final)

df_spark_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_CATALOG}.balanco_energia_subsistema")

In [0]:
# ===================================================================
# Summary and Verification
# ===================================================================

# Verify each table
tables = [
    'pld_historico_semanal',
    'pld_horario',
    'balanco_energia_subsistema'
]

for table_name in tables:
    df = spark.table(f"{SILVER_CATALOG}.{table_name}")
    count = df.count()
    display(df.limit(5))

submercado,mes_referencia,periodo_comercializacao,data,hora,pld_hora,processed_timestamp,data_quality_flag
SUL,2004-10-01T00:00:00.000Z,529,2004-10-23T00:00:00.000Z,0,18.59,2026-08-29T19:42:57.310Z,VALID
SUL,2004-10-01T00:00:00.000Z,697,2004-10-30T00:00:00.000Z,0,18.59,2026-08-29T19:42:57.310Z,VALID
NORDESTE,2004-11-01T00:00:00.000Z,121,2004-11-06T00:00:00.000Z,0,18.59,2026-08-29T19:42:57.310Z,VALID
NORDESTE,2004-11-01T00:00:00.000Z,289,2004-11-13T00:00:00.000Z,0,18.59,2026-08-29T19:42:57.310Z,VALID
NORDESTE,2004-11-01T00:00:00.000Z,457,2004-11-20T00:00:00.000Z,0,18.59,2026-08-29T19:42:57.310Z,VALID


submercado,mes_referencia,periodo_comercializacao,data,hora,pld_hora,ano_arquivo,processed_timestamp,data_quality_flag
NORDESTE,2024-03-01T00:00:00.000Z,61,2024-03-03T00:00:00.000Z,12,61.07,2024,2026-08-29T19:43:11.881Z,VALID
NORTE,2024-03-01T00:00:00.000Z,61,2024-03-03T00:00:00.000Z,12,61.07,2024,2026-08-29T19:43:11.881Z,VALID
SUDESTE,2024-03-01T00:00:00.000Z,61,2024-03-03T00:00:00.000Z,12,61.07,2024,2026-08-29T19:43:11.881Z,VALID
SUL,2024-03-01T00:00:00.000Z,61,2024-03-03T00:00:00.000Z,12,61.07,2024,2026-08-29T19:43:11.881Z,VALID
NORDESTE,2024-03-01T00:00:00.000Z,62,2024-03-03T00:00:00.000Z,13,61.07,2024,2026-08-29T19:43:11.881Z,VALID


subsistema_id,subsistema_nome,timestamp,geracao_hidraulica_mw,geracao_termica_mw,geracao_eolica_mw,geracao_solar_mw,geracao_total_mw,pct_hidraulica,pct_termica,pct_eolica,pct_solar,carga_mw,intercambio_mw,ano_arquivo,processed_timestamp,data_quality_flag
SIN,SISTEMA INTERLIGADO NACIONAL,2003-09-01T23:00:00.000Z,37237.8,3068.18,0.0,0.0,40305.98,92.38777967934286,7.612220320657133,0.0,0.0,40352.28,-46.3,2003,2026-08-29T19:44:10.994Z,VALID
SE,SUDESTE/CENTRO-OESTE,2003-09-01T23:00:00.000Z,26666.89999999,2169.23999999,0.0,0.0,28836.13999998,92.47735653942759,7.522643460572407,0.0,0.0,25086.38001464,3749.75998535,2003,2026-08-29T19:44:10.994Z,VALID
S,SUL,2003-09-01T23:00:00.000Z,3530.7,820.28,0.0,0.0,4350.98,81.1472357951542,18.852764204845805,0.0,0.0,6493.93998535,-2142.95998535,2003,2026-08-29T19:44:10.994Z,VALID
NE,NORDESTE,2003-09-02T00:00:00.000Z,3951.2,78.67999999,0.0,0.0,4029.8799999899998,98.04758454370366,1.9524154562963474,0.0,0.0,5774.78,-1744.9,2003,2026-08-29T19:44:10.994Z,VALID
N,NORTE,2003-09-02T00:00:00.000Z,2703.0,0.0,0.0,0.0,2703.0,100.0,0.0,0.0,0.0,2571.3,131.7,2003,2026-08-29T19:44:10.994Z,VALID
